<a href="https://colab.research.google.com/github/fatimaezzahrabakas61-web/meteo_project/blob/main/kpis_calcule_marketing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import pandas as pd

def load_data(path):
    df = pd.read_csv(path)
    return df


def preprocess_data(df):
    # Conversion des dates
    date_cols = ["date_served", "date_subscribed", "date_canceled"]
    for col in date_cols:
        df[col] = pd.to_datetime(df[col], errors='coerce')

    # Normalisation booléens
    df["converted"] = df["converted"].astype(bool)
    df["is_retained"] = df["is_retained"].astype(bool)

    return df

KPI : Taux de conversion

In [21]:
def conversion_rate(df):
    total_users = df["user_id"].nunique()
    converted_users = df[df["converted"] == True]["user_id"].nunique()

    return converted_users / total_users if total_users > 0 else 0

KPI : Taux de rétention

In [22]:
def retention_rate(df):
    total_users = df["user_id"].nunique()
    retained_users = df[df["is_retained"] == True]["user_id"].nunique()

    return retained_users / total_users if total_users > 0 else 0

KPI : Taux d’abonnement

In [23]:
def subscription_rate(df):
    total_users = df["user_id"].nunique()
    subscribed_users = df[df["date_subscribed"].notna()]["user_id"].nunique()

    return subscribed_users / total_users if total_users > 0 else 0


KPI : Churn rate



In [24]:
def churn_rate(df):
    subscribed = df[df["date_subscribed"].notna()]["user_id"].nunique()
    churned = df[df["date_canceled"].notna()]["user_id"].nunique()

    return churned / subscribed if subscribed > 0 else 0

Performance par canal marketing

In [16]:
def performance_by_channel(df):
    result = (
        df.groupby("marketing_channel")
        .agg(
            users=("user_id", "nunique"),
            conversions=("converted", "sum")
        )
    )

    result["conversion_rate"] = result["conversions"] / result["users"]
    return result.reset_index()

Performance par segment d’âge

In [17]:
def performance_by_age(df):
    result = (
        df.groupby("age_group")
        .agg(
            users=("user_id", "nunique"),
            conversions=("converted", "sum")
        )
    )

    result["conversion_rate"] = result["conversions"] / result["users"]
    return result.reset_index()

Analyse A/B test (variant)

In [18]:
def performance_by_variant(df):
    result = (
        df.groupby("variant")
        .agg(
            users=("user_id", "nunique"),
            conversions=("converted", "sum")
        )
    )

    result["conversion_rate"] = result["conversions"] / result["users"]
    return result.reset_index()

In [19]:
def run_kpi_pipeline(path):
    df = load_data(path)
    df = preprocess_data(df)

    results = {
        "conversion_rate": conversion_rate(df),
        "retention_rate": retention_rate(df),
        "subscription_rate": subscription_rate(df),
        "churn_rate": churn_rate(df),
        "by_channel": performance_by_channel(df),
        "by_age": performance_by_age(df),
        "by_variant": performance_by_variant(df)
    }

    return results

Exemple d'un utilisateur

In [25]:
results = run_kpi_pipeline("marketing_clean.csv")

print("Conversion rate:", results["conversion_rate"])
print("Retention rate:", results["retention_rate"])
print("Subscription rate:", results["subscription_rate"])
print("Churn rate:", results["churn_rate"])

print(results["by_channel"])
print(results["by_age"])
print(results["by_variant"])

Conversion rate: 0.13886988644137366
Retention rate: 0.0927623477903954
Subscription rate: 0.13955397455192228
Churn rate: 0.3431372549019608
  marketing_channel  users  conversions  conversion_rate
0             Email    553          193         0.349005
1          Facebook   1758          237         0.134812
2         House Ads   3968          298         0.075101
3         Instagram   1740          265         0.152299
4              Push    972           83         0.085391
5           Unknown     15            0         0.000000
  age_group  users  conversions  conversion_rate
0      0-18   1208          201         0.166391
1     19-24   1307          314         0.240245
2     24-30   1220          238         0.195082
3     30-36   1059           82         0.077432
4     36-45   1058           78         0.073724
5     45-55   1058           80         0.075614
6       55+    981           83         0.084608
           variant  users  conversions  conversion_rate
0          